In [17]:
# --- Importações Essenciais ---
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.panel import Panel

# --- Adiciona a pasta 'backend' ao caminho do Python ---
# Isto permite que o notebook encontre os teus módulos da Shaula
sys.path.append('..')

# --- Importa os Módulos da Shaula ---
from backend.agente import AgenteReflexivo
from backend.gerenciador_usuarios import GerenciadorDeUsuarios
from backend.analisador_de_dados import AnalisadorDeDados
from backend.main import obter_resposta_llm # Precisamos da função que chama a OpenAI

# --- Configurações Visuais ---
sns.set_style("whitegrid")
console = Console()

print("✅ Ambiente e bibliotecas carregados com sucesso.")

✅ Ambiente e bibliotecas carregados com sucesso.


In [18]:
# --- Inicializa a Shaula para a nossa sessão de pesquisa ---

# 1. Cria o gestor de utilizadores e define o utilizador atual
gerenciador_de_usuarios = GerenciadorDeUsuarios(caminho_arquivo="../data/usuarios.json")
usuario_atual = gerenciador_de_usuarios.obter_ou_criar_usuario_atual("Abraão")

# 2. Cria a instância principal do agente reflexivo
agente = AgenteReflexivo(
    usuario_atual=usuario_atual, 
    gerenciador=gerenciador_de_usuarios, 
    console_log=console
)

# 3. Adiciona a função de chamada da LLM ao agente para que o Analisador a possa usar
agente.obter_resposta_llm = obter_resposta_llm

# 4. Adiciona os prompts genéricos ao agente para que o Analisador os possa aceder
# (Copia e cola os teus 4 prompts genéricos aqui)
agente.prompts_analise = {
    "passo_1_avaliacao_inicial": """
Você é uma cientista de dados Sênior e especialista em análise exploratória. Acabou de receber um novo dataset para um projeto. O seu objetivo é prever a coluna '{nome_da_coluna_alvo}'.

**Contexto:**
Abaixo estão os resultados dos comandos `.info()` e `.describe()` executados no dataset.

**Resultado do `.info()`:**
{resultado_info}


**Resultado do `.describe()`:**
{resultado_describe}


**Tarefa de Análise Crítica:**
Com base **apenas** nestas informações, forneça uma avaliação inicial completa e estruturada:

1.  **Tipo de Problema:** Esta é uma tarefa de **Regressão** ou **Classificação**? Justifique a sua resposta com base na natureza (tipo de dado, número de valores únicos) da coluna-alvo '{nome_da_coluna_alvo}'.

2.  **Qualidade dos Dados (Primeira Impressão):** Identifique os 3 principais desafios de pré-processamento que você prevê. Foque em:
    * **Dados Ausentes:** Quais colunas têm valores nulos e isso parece ser um problema significativo?
    * **Tipos de Dados:** Existem colunas que precisam de conversão (ex: 'Object' para data ou número)?
    * **Escalas Numéricas:** As colunas numéricas parecem ter escalas muito diferentes (ex: uma vai de 0 a 1 e outra de 0 a 1.000.000)?
    * **Cardinalidade Categórica:** Existem colunas de texto? Se sim, parecem ter muitas categorias únicas?

3.  **Hipótese Inicial:** Qual a sua primeira hipótese sobre o que será mais desafiador neste projeto (ex: "o feature engineering será complexo devido à falta de preditores óbvios", ou "a limpeza de dados será a fase mais demorada devido à quantidade de valores ausentes").

4.  **Sugestão de Próximo Passo:** Confirme que o próximo passo lógico é uma Análise Exploratória de Dados (AED) mais profunda para visualizar as distribuições e correlações.
""",

            "passo_2_plano_aed": """
Shaula, agora que temos a avaliação inicial, a tua tarefa é delinear um plano de ação para a Análise Exploratória de Dados (AED).

**Contexto:**
- O nosso problema é de **{tipo_de_problema}**.
- A nossa variável-alvo é **'{nome_da_coluna_alvo}'**.
- As colunas numéricas candidatas a preditores são: {lista_de_colunas_numericas}.
- As colunas categóricas candidatas a preditores são: {lista_de_colunas_categoricas}.

**Tarefa: Criar um Plano de AED**
Descreve, passo a passo, o plano que seguirias. Para cada passo, especifica qual a tua principal pergunta e que tipo de gráfico usarias para a responder. O teu plano deve cobrir:

1.  **Análise da Variável-Alvo:** Como investigarias a distribuição da coluna '{nome_da_coluna_alvo}'? Que problema específico (ex: assimetria, desbalanceamento de classes) estás a procurar?

2.  **Análise de Preditores Numéricos:** Como investigarias a relação entre as features numéricas e a variável-alvo? Qual é a ferramenta estatística principal que usarias?

3.  **Análise de Preditores Categóricos:** Como investigarias a relação entre as features categóricas e a variável-alvo? Que tipo de visualização seria mais eficaz?

4.  **Conclusão e Próximo Passo:** Com base neste plano, qual é o *insight* mais importante que esperas obter da AED?""",

            "passo_3_estrategia_pipeline": """
Shaula, a Análise Exploratória de Dados foi concluída. Agora, a tua tarefa como Engenheira de Machine Learning é projetar um pipeline de pré-processamento robusto e completo com o Scikit-Learn.

**Contexto (Achados da AED):**
- {resumo_dos_achados_da_aed} 
(Ex: "A variável-alvo está altamente desbalanceada. As features numéricas 'A' e 'B' têm uma distribuição assimétrica. A feature categórica 'C' tem 5% de valores ausentes.")

**Tarefa: Projetar o Pipeline de Pré-processamento**
Descreve, de forma estruturada, o teu plano para construir um `ColumnTransformer` que prepare os dados para o modelo. Justifica cada escolha.

1.  **Estratégia de Divisão de Dados:** Como dividirias os dados em treino e teste? Que parâmetro específico usarias na função `train_test_split` para lidar com o desbalanceamento que encontrámos?

2.  **Pipeline para Features Numéricas:** Descreve a sequência de etapas (transformadores do Scikit-Learn) que aplicarias a todas as colunas numéricas.

3.  **Pipeline para Features Categóricas:** Descreve a sequência de etapas que aplicarias a todas as colunas categóricas.

4.  **Tratamentos Especiais (Se necessário):** Com base nos achados da AED, propões algum tratamento especial para colunas específicas (ex: uma transformação logarítmica para as colunas assimétricas)? Como integrarias isso no pipeline?""",
           
            "passo_4_analise_performance": """
Shaula, executámos o pipeline e treinámos um modelo de baseline ({nome_do_modelo}) para a nossa tarefa de {tipo_de_problema}. A tua tarefa final é realizar uma análise crítica e profunda da sua performance.

**Contexto (Resultados do Modelo):**

**Matriz de Confusão:**
{matriz_de_confusao}


**Relatório de Classificação:**
{relatorio_de_classificacao}


**Tarefa: Análise Crítica e Sugestão Estratégica**
Fornece uma análise completa dos resultados:

1.  **Interpretação das Métricas:** Explica o que as métricas (Precision, Recall, F1-Score) para cada classe significam no contexto do nosso problema. Qual métrica consideras a mais importante aqui e porquê?

2.  **Análise dos Erros:** Com base na Matriz de Confusão, qual é o tipo de erro mais comum que o modelo está a cometer (Falsos Positivos ou Falsos Negativos)? Qual é o impacto disso no problema de negócio?

3.  **Conclusão Geral:** Este modelo de baseline é "bom" o suficiente? Ele resolve o problema principal? Justifica.

4.  **Sugestão Estratégica:** Com base nesta análise, qual é a tua recomendação para o **próximo passo**? ex: "tentar um modelo mais complexo", "focar em feature engineering", "usar técnicas de reamostragem como SMOTE para tratar o desbalanceamento", etc.).
"""
}


print(f"✅ Agente Shaula instanciado e pronto para o utilizador: {agente.usuario_atual.nome}")

Perfil encontrado. Bem-vindo de volta, Abraão!

Estado inicializado para Abraão. Memória com 0 registos.

✅ Agente Shaula instanciado e pronto para o utilizador: Abraão


In [19]:
# --- Esta célula contém a lógica de preparação de dados específica para o Olist ---
# Ela transforma os 9 ficheiros brutos na tabela de análise 'ord_base'

DATA_PATH = '../data/olist_ecommerce/'

try:
    # Carrega todos os datasets necessários
    orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))
    items = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_items_dataset.csv'))
    payments = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
    reviews = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_reviews_dataset.csv'))
    customers = pd.read_csv(os.path.join(DATA_PATH, 'olist_customers_dataset.csv'))
    products = pd.read_csv(os.path.join(DATA_PATH, 'olist_products_dataset.csv'))
    sellers = pd.read_csv(os.path.join(DATA_PATH, 'olist_sellers_dataset.csv'))
    
    # Executa todo o processo de fusão e limpeza
    date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
    for c in date_cols:
        if c in orders.columns:
            orders[c] = pd.to_datetime(orders[c], errors='coerce')
    if 'review_creation_date' in reviews.columns:
        reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
    
    delivered = orders[orders['order_status'].isin(['delivered'])].copy()
    item_agg = items.groupby('order_id', as_index=False).agg(revenue=('price', 'sum'), freight=('freight_value', 'sum'), n_items=('order_item_id','count'))
    pay_agg = payments.groupby('order_id', as_index=False).agg(payment_total=('payment_value','sum'))
    rev_sorted = reviews.sort_values('review_creation_date')
    rev_latest = rev_sorted.groupby('order_id', as_index=False).tail(1)[['order_id','review_score']]
    cust_slim = customers[['customer_id','customer_unique_id','customer_state']].drop_duplicates('customer_id')

    ord_base = (delivered.merge(item_agg, on='order_id', how='left').merge(pay_agg, on='order_id', how='left').merge(rev_latest, on='order_id', how='left').merge(cust_slim, on='customer_id', how='left'))
    
    print("✅ Tabela de análise 'ord_base' criada com sucesso.")
    display(ord_base.head(3))

except FileNotFoundError as e:
    print(f"❌ ERRO: {e}")

❌ ERRO: [Errno 2] No such file or directory: '../data/olist_ecommerce/olist_orders_dataset.csv'


In [20]:
# --- Agora, vamos entregar os dados para a Shaula e iniciar o fluxo de análise ---

# 1. Define o DataFrame e a coluna-alvo para o experimento
dataframe_para_analise = ord_base
alvo = 'review_score'

# 2. Inicia o modo de análise no agente, passando os dados
print("🚀 A iniciar o modo de análise de dados com a Shaula...")
agente.sessao_de_analise = AnalisadorDeDados(agente, dataframe=dataframe_para_analise, coluna_alvo=alvo)

# 3. Pede à Shaula para começar (primeira interação)
resposta_json, raciocinio = agente.sessao_de_analise.iniciar_fluxo()

# 4. Extrai e exibe a primeira mensagem da Shaula
prompt_para_llm = json.loads(resposta_json)["parametros"]["prompt"]
resposta_shaula = obter_resposta_llm(prompt_para_llm).get("conteudo")

console.print(Panel(resposta_shaula, title="[cyan]Shaula diz:[/cyan]", border_style="cyan"))

🚀 A iniciar o modo de análise de dados com a Shaula...


TypeError: AnalisadorDeDados.__init__() got an unexpected keyword argument 'dataframe'

In [ ]:
# --- Para continuar o fluxo, envia uma confirmação para a Shaula ---

resposta_json, raciocinio = agente.sessao_de_analise.continuar_fluxo("sim, pode continuar")

# Extrai e exibe a próxima mensagem da Shaula
prompt_para_llm = json.loads(resposta_json)["parametros"]["prompt"]
resposta_shaula = obter_resposta_llm(prompt_para_llm).get("conteudo")

console.print(Panel(resposta_shaula, title="[cyan]Shaula diz:[/cyan]", border_style="cyan"))